# Paper 2 — Answer-Parser Fix + Ground-Truth Repair

**Runtime: CPU, minutes. Needs internet (re-downloads the MATH dataset).**

Two corrections that must both land before notebook 6 can report corrected numbers. Run this after generation (notebooks 3 and 4) and before verification (notebook 6).

**Part 1 — the parser.** Three confirmed bugs in `extract_answer`: nested braces truncating at `\dfrac{1`, a prose fallback returning words like `'the'` and `'approximately'`, and no normalisation so `'864.'` and `'864'` compared unequal. The fix changed 7–9% of rollout verdicts.

**Part 2 — the ground truth itself.** Worse than the parser: `answer_gt` for 71 of 300 problems had been *written* by the buggy parser, so the reference answers were truncated too. This re-derives them from the source dataset, joining on normalised problem text.

The audit found 195 flags changed — 184 false negatives and 11 false positives, with **zero false positives in any analysis pool.**

Writes `repaired_ground_truth.json`, which notebook 6 requires.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

> **Run top to bottom, both parts.** Each part is a separate script and both define a function called `main()`, so part 2 redefines part 1's. That is correct when run in order — part 1 has already executed by then — but re-running part 1's call cell *after* part 2 would run part 2's `main()` instead. Re-run from the top if you need to redo part 1.

Part 2 matters more than part 1: a bad parser mis-scores rollouts, but a bad `answer_gt` means the reference answers themselves were wrong.


## Part 1 — answer-parser fix (with its test cases)


In [ ]:
# =============================================================================
# ANSWER EXTRACTION FIX + RE-ANALYSIS
#
# WHY
# The original extract_answer (verbatim from Task_1_Data_Pipeline.ipynb) fails
# on three distinct patterns, all confirmed against real run output:
#
#   1. NESTED BRACES. The boxed regex stops at the first closing brace, so a
#      boxed dfrac{1}{2} extracts only "dfrac{1". This affects every
#      fractional, radical or subscripted answer.
#   2. FALLBACK GRABS WORDS. The alternation matches "Therefore" before
#      "answer is", then takes the next token -- so "Therefore the answer is 7"
#      extracts "the". Likewise "answer is approximately 12.1" gives
#      "approximately".
#   3. NO NORMALISATION. "864." does not equal "864"; "50 " does not equal "50".
#
# Baseline (UNSTEERED) extraction success was only 56-69%, so a large share of
# failures are the parser rather than suppression. Every necessity number
# inherits that noise.
#
# THIS SCRIPT DOES NOT SILENTLY REPLACE ANY NUMBER.
# It reports old vs new side by side on saved rollout text so the change can be
# inspected first. A parser fix that moves conclusions has to be visibly an
# improvement, not merely a change.
#
# COST: CPU only, seconds. All rollout text is already on disk.
# =============================================================================



from pathlib import Path
import json
import re

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"

In [ ]:
EVENTS_REV = EXPORT_DIR / "counterfactual_pilot_events.json"
EVENTS_FIL = EXPORT_DIR / "filler_control_pilot_events.json"
REPORT_OUT = EXPORT_DIR / "extraction_fix_report.json"

ROLLOUT_FILES = [
    ("steering revision", "steering_revision_rollouts_powered.json"),
    ("steering filler", "steering_filler_rollouts_powered.json"),
    ("lexical revision", "counterfactual_rollout_results.json"),
    ("lexical filler", "filler_control_rollout_results.json"),
]


# ------------------------------------------------------------------ OLD -----
def extract_answer_old(text):
    """Verbatim original. Kept only for comparison -- do not modify."""
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)',
                  text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"

In [ ]:
# ------------------------------------------------------------------ NEW -----
def boxed_spans(text):
    r"""Return the contents of every \boxed{...} using brace COUNTING rather
    than a non-greedy character class, so nested braces survive intact."""
    spans = []
    for m in re.finditer(r'\\boxed\s*\{', text):
        start = m.end() - 1
        depth = 0
        for j in range(start, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                depth -= 1
                if depth == 0:
                    spans.append(text[start + 1:j])
                    break
    return spans


# Tokens the fallback must never return as an answer.
STOPWORDS = {
    "the", "a", "an", "is", "approximately", "about", "roughly", "exactly",
    "correct", "incorrect", "therefore", "thus", "so", "then", "that", "this",
    "it", "equal", "equals", "our", "final", "answer", "value", "result",
    "less", "more", "different", "same",
}


def normalise_answer(s):
    r"""Conservative canonicalisation. Strips wrappers, whitespace and trailing
    punctuation, and unifies a few LaTeX spellings. Deliberately does NOT
    attempt semantic equivalence -- it must never merge genuinely different
    answers."""
    if s is None:
        return "UNKNOWN"
    s = s.strip()
    s = re.sub(r'^\$+|\$+$', '', s)
    s = re.sub(r'\\(?:left|right|,|;|!|quad|qquad)', '', s)
    s = re.sub(r'\\text\s*\{([^}]*)\}', r'\1', s)
    s = s.replace(r'\dfrac', r'\frac').replace(r'\tfrac', r'\frac')
    s = re.sub(r'\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}', r'\1/\2', s)
    s = re.sub(r'\s+', '', s)
    s = s.rstrip('.,;:')
    s = re.sub(r'^\{|\}$', '', s)
    return s or "UNKNOWN"


def extract_answer_new(text):
    boxed = boxed_spans(text)
    if boxed:
        return normalise_answer(boxed[-1])

    patterns = (
        r'(?:final\s+)?answer\s*(?:is|=|:)\s*(.{0,40})',
        r'(?:therefore|thus|hence)\s*,?\s*(.{0,40})',
    )
    for pat in patterns:
        for m in re.finditer(pat, text, re.IGNORECASE):
            for tok in re.findall(r'[-+]?\$?\\?[0-9A-Za-z\\{}\^_/\.]+', m.group(1)):
                cand = normalise_answer(tok)
                if cand == "UNKNOWN" or cand.lower() in STOPWORDS:
                    continue
                # must contain a digit or a LaTeX command, else it is prose
                if re.search(r'[0-9]|\\', cand):
                    return cand
    return "UNKNOWN"


def answers_match(pred, gt):
    if pred == "UNKNOWN":
        return False
    return normalise_answer(pred) == normalise_answer(gt)

In [ ]:
# ------------------------------------------------------------- analysis -----
def main():
    ground_truth = {}
    for path in (EVENTS_REV, EVENTS_FIL):
        if path.exists():
            for e in json.load(open(path)):
                ground_truth[(e["model"], e["problem_id"])] = e["answer_gt"]

    print("\n" + "#" * 78)
    print("# EXTRACTION FIX -- old vs new on saved rollout text")
    print("#" * 78)

    report = {}
    for label, fname in ROLLOUT_FILES:
        path = EXPORT_DIR / fname
        if not path.exists():
            print(f"\n  {label}: file not found, skipping")
            continue

        n_ro = n_unk_old = n_unk_new = n_cor_old = n_cor_new = 0
        changed = 0
        examples = []

        for r in json.load(open(path)):
            gt = ground_truth.get((r["model"], r["problem_id"]))
            for t in r.get("rollout_texts", []):
                n_ro += 1
                a_old = extract_answer_old(t)
                a_new = extract_answer_new(t)
                if a_old == "UNKNOWN":
                    n_unk_old += 1
                if a_new == "UNKNOWN":
                    n_unk_new += 1
                if gt is None:
                    continue
                c_old = a_old.strip() == gt.strip()
                c_new = answers_match(a_new, gt)
                n_cor_old += c_old
                n_cor_new += c_new
                if c_old != c_new:
                    changed += 1
                    if len(examples) < 8:
                        examples.append((a_old, a_new, gt, bool(c_old), bool(c_new)))

        if n_ro == 0:
            print(f"\n  {label}: no rollout text saved, skipping")
            continue

        print(f"\n  {label}  ({n_ro} rollouts)")
        print(f"    UNKNOWN : {n_unk_old:>4} ({n_unk_old / n_ro:>4.0%})  ->  "
              f"{n_unk_new:>4} ({n_unk_new / n_ro:>4.0%})")
        print(f"    correct : {n_cor_old:>4} ({n_cor_old / n_ro:>4.0%})  ->  "
              f"{n_cor_new:>4} ({n_cor_new / n_ro:>4.0%})")
        print(f"    verdict changed on {changed} rollouts ({changed / n_ro:.1%})")
        for a_old, a_new, gt, c_old, c_new in examples:
            tag = "recovered" if (c_new and not c_old) else "LOST"
            print(f"      {tag:<10} old={a_old!r:<20} new={a_new!r:<20} gt={gt!r}")

        report[label] = {
            "n_rollouts": n_ro,
            "unknown_old": n_unk_old, "unknown_new": n_unk_new,
            "correct_old": n_cor_old, "correct_new": n_cor_new,
            "verdict_changed": changed,
        }

    print("\n" + "#" * 78)
    print("# READ")
    print("#" * 78)
    regressed = [k for k, v in report.items() if v["correct_new"] < v["correct_old"]]
    if regressed:
        for k in regressed:
            v = report[k]
            print(f"  WARNING: {k} lost correct answers "
                  f"({v['correct_old']} -> {v['correct_new']}). Investigate before use.")
    else:
        print("  The new parser recovers answers in every arm and loses none.")
        print("  Any LOST examples above are cases where the old parser matched by")
        print("  accident -- typically both prediction and ground truth truncated the")
        print("  same way. Worth eyeballing, but a net gain with no arm regressing is")
        print("  the expected signature of a genuine fix.")
    print("\n  NEXT: rerun the necessity analyses using extract_answer_new and check")
    print("  whether the validity result moves. Do not assume it survives -- the")
    print("  reason for fixing the parser is that the old numbers were noisy, and")
    print("  that cuts both ways.")

    json.dump(report, open(REPORT_OUT, "w"), indent=2)
    print(f"\nWrote {REPORT_OUT}")


if __name__ == "__main__":
    main()

## Part 2 — ground-truth repair


In [ ]:
# =============================================================================
# GROUND-TRUTH REPAIR AND CORRECTNESS AUDIT
#
# THE PROBLEM
# answer_gt was written at Stage 1 using the old parser, whose boxed regex
# stops at the first closing brace. Any answer containing braces was stored
# truncated: \dfrac{1}{3} became "\dfrac{1". This affects the SAME 71 of 300
# problems in every model -- it is a property of the problems, not the models.
#
# The `correct` flag was then computed by comparing a truncated prediction
# against a truncated target. That can agree by accident (both cut to the same
# prefix) even when the underlying answers differ, e.g. \dfrac{1}{3} and
# \dfrac{1}{2} both reduce to "\dfrac{1". So `correct` is unreliable for those
# 71 problems -- and `correct` is the eligibility filter used by every
# necessity event-selection script.
#
# The trace files contain no independent copy of the dataset answer (verified:
# 71/71 have no clean gt field), so the only sound source is the dataset
# itself.
#
# WHAT THIS DOES
#   1. Loads EleutherAI/hendrycks_math and indexes canonical answers by
#      problem text, extracting with brace-counting.
#   2. Joins to the local traces on normalised problem text.
#   3. Recomputes correctness properly and reports how many flags CHANGE.
#   4. Writes a repaired ground-truth map. It does NOT modify the traces --
#      the audit comes first.
#
# READ:
#   Few flags change -> existing event selection stands; fix is forward-looking.
#   Many change      -> event pools were selected on bad flags and downstream
#                       necessity results need revisiting.
#
# COST: CPU + internet for the dataset. Minutes.
# =============================================================================



from pathlib import Path
from collections import Counter, defaultdict
import json
import re

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")

In [ ]:
TRACES_DIR = GDRIVE_BASE / "traces"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
REPAIRED_OUT = EXPORT_DIR / "repaired_ground_truth.json"
AUDIT_OUT = EXPORT_DIR / "correctness_audit.json"

HENDRYCKS_CONFIGS = [
    "algebra", "counting_and_probability", "geometry", "intermediate_algebra",
    "number_theory", "prealgebra", "precalculus",
]

In [ ]:
def boxed_spans(text):
    r"""Contents of every \boxed{...}, via brace counting so nesting survives."""
    spans = []
    for m in re.finditer(r'\\boxed\s*\{', text or ""):
        start = m.end() - 1
        depth = 0
        for j in range(start, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                depth -= 1
                if depth == 0:
                    spans.append(text[start + 1:j])
                    break
    return spans


def normalise_answer(s):
    r"""Conservative canonicalisation -- must never merge distinct answers."""
    if s is None:
        return "UNKNOWN"
    s = s.strip()
    s = re.sub(r'^\$+|\$+$', '', s)
    s = re.sub(r'\\(?:left|right|,|;|!|quad|qquad)', '', s)
    s = re.sub(r'\\text\s*\{([^}]*)\}', r'\1', s)
    s = s.replace(r'\dfrac', r'\frac').replace(r'\tfrac', r'\frac')
    s = re.sub(r'\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}', r'\1/\2', s)
    s = re.sub(r'\s+', '', s)
    s = s.rstrip('.,;:')
    s = re.sub(r'^\{|\}$', '', s)
    return s or "UNKNOWN"


def normalise_problem(p):
    """Join key. Whitespace and case are unreliable across sources; the
    mathematical content is not."""
    return re.sub(r'\s+', ' ', (p or "").strip()).lower()


def looks_truncated(s):
    return isinstance(s, str) and s.count('{') != s.count('}')


def build_dataset_index():
    from datasets import load_dataset
    index = {}
    for cfg in HENDRYCKS_CONFIGS:
        print(f"  loading {cfg} ...")
        for split in ("train", "test"):
            try:
                ds = load_dataset("EleutherAI/hendrycks_math", cfg, split=split,
                                  trust_remote_code=False)
            except Exception as e:
                print(f"    {split}: skipped ({e})")
                continue
            for row in ds:
                sol = row.get("solution", "")
                b = boxed_spans(sol)
                if not b:
                    continue
                index[normalise_problem(row.get("problem", ""))] = {
                    "answer_raw": b[-1],
                    "answer_norm": normalise_answer(b[-1]),
                    "level": row.get("level"), "subject": row.get("type") or cfg,
                }
    print(f"  indexed {len(index)} problems from the dataset")
    return index

In [ ]:
def main():
    print("\n" + "#" * 78)
    print("# GROUND-TRUTH REPAIR AND CORRECTNESS AUDIT")
    print("#" * 78)

    print("\nBuilding dataset index...")
    index = build_dataset_index()

    repaired = {}
    audit = defaultdict(Counter)
    examples = []

    for model_dir in sorted(TRACES_DIR.iterdir()):
        if not model_dir.is_dir():
            continue
        model = model_dir.name
        for f in sorted(model_dir.glob("*.json")):
            try:
                o = json.load(open(f))
            except Exception:
                continue
            pid = o.get("problem_id")
            if pid is None or "problem" not in o:
                continue

            key = normalise_problem(o["problem"])
            hit = index.get(key)
            a = audit[model]
            a["total"] += 1

            if hit is None:
                a["unmatched"] += 1
                continue
            a["matched"] += 1

            stored_gt = o.get("answer_gt", "")
            true_gt = hit["answer_norm"]
            if looks_truncated(stored_gt):
                a["gt_was_truncated"] += 1

            # the model's own answer, properly re-extracted
            pred = boxed_spans(o.get("full_response", ""))
            pred_norm = normalise_answer(pred[-1]) if pred else "UNKNOWN"

            old_flag = bool(o.get("correct"))
            new_flag = (pred_norm != "UNKNOWN") and (pred_norm == true_gt)

            if old_flag == new_flag:
                a["flag_unchanged"] += 1
            else:
                a["flag_CHANGED"] += 1
                a["false_positive" if old_flag else "false_negative"] += 1
                if len(examples) < 12:
                    examples.append({
                        "model": model, "problem_id": pid,
                        "stored_gt": stored_gt, "true_gt": true_gt,
                        "pred": pred_norm, "old": old_flag, "new": new_flag,
                    })

            repaired[f"{model}|{pid}"] = {
                "stored_answer_gt": stored_gt, "true_answer_gt": true_gt,
                "pred_normalised": pred_norm,
                "correct_old": old_flag, "correct_new": new_flag,
            }

    print("\n" + "=" * 78)
    print("PER-MODEL AUDIT")
    print("=" * 78)
    hdr = f"  {'Model':<32}{'matched':>8}{'unmatch':>8}{'trunc gt':>9}{'CHANGED':>9}{'FP':>5}{'FN':>5}"
    print(hdr)
    total_changed = 0
    for model, a in sorted(audit.items()):
        total_changed += a["flag_CHANGED"]
        print(f"  {model[-31:]:<32}{a['matched']:>8}{a['unmatched']:>8}"
              f"{a['gt_was_truncated']:>9}{a['flag_CHANGED']:>9}"
              f"{a['false_positive']:>5}{a['false_negative']:>5}")

    if examples:
        print("\n  examples of changed verdicts:")
        for e in examples:
            print(f"    {e['model'][-14:]:<15} {e['problem_id']:<12} "
                  f"stored={e['stored_gt']!r:<16} true={e['true_gt']!r:<16} "
                  f"pred={e['pred']!r:<16} {e['old']} -> {e['new']}")

    print("\n" + "#" * 78)
    print("# READ")
    print("#" * 78)
    any_unmatched = sum(a["unmatched"] for a in audit.values())
    if any_unmatched:
        print(f"  {any_unmatched} problems did not match the dataset by text. If this is")
        print("  more than a handful, the join key needs work before trusting the audit.")
    if total_changed == 0:
        print("  NO correctness flags change. The truncation was cosmetic: predictions and")
        print("  targets were cut identically and the verdicts happened to be right.")
        print("  Existing event selection stands. Fix the parser going forward and note")
        print("  the issue in limitations; no re-run needed.")
    else:
        print(f"  {total_changed} correctness flags CHANGE across all models.")
        print("  `correct` gates eligibility in every necessity event-selection script, so")
        print("  the event pools were built on some wrong flags. Assess whether the")
        print("  affected problems actually entered the pools before deciding on a re-run --")
        print("  a changed flag only matters if that problem was selected.")
        print("  False positives (was 'correct', actually wrong) are the damaging direction:")
        print("  those problems should never have been eligible.")

    json.dump(repaired, open(REPAIRED_OUT, "w"), indent=2)
    json.dump({m: dict(a) for m, a in audit.items()}, open(AUDIT_OUT, "w"), indent=2)
    print(f"\n  Wrote {REPAIRED_OUT}")
    print(f"  Wrote {AUDIT_OUT}")
    print("\n  NOTE: traces were NOT modified. This is an audit; repair the pipeline")
    print("  only once the scale of the problem is understood.")


if __name__ == "__main__":
    main()